In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
from statsmodels.api import OLS
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df=pd.read_csv('/kaggle/input/student-performance-dataset/ultimate_student_productivity_dataset_5000.csv')

In [ ]:
df.head() #Check the first five rows of the dataset

In [ ]:
df.info() #check the datatypes of the dataset

In [ ]:
df.drop(['student_id'],axis=1,inplace=True) #drop unnecesary column

In [ ]:
df.describe() #Statistical Summary Calculation

In [ ]:
df.isnull().sum() # Check the null values of the dataset

In [ ]:
counts = df['gender'].value_counts()
plt.figure(figsize=(8,6))
plt.pie(
    counts.values,
    labels=counts.index,   # Proper labels
    autopct='%.1f%%',
    startangle=90
)
plt.title("Gender Distribution")
plt.axis('equal')  # Makes it a perfect circle
plt.show()

In [ ]:
df['academic_level'].value_counts() #Count of different academic level

In [ ]:
plt.figure(figsize=(10,6)) # set the figure size
sns.countplot(data=df,x='internet_quality',palette='magma') #Counplot of internet_quality
plt.title('Counplot of internet quality') #Title of the plot
plt.xlabel('Internet_Quality') #level of x-axis
plt.ylabel('Count') #level of y-axis
plt.show()

In [ ]:
plt.figure(figsize=(10,6)) # set the figure size
sns.countplot(data=df,x='academic_level',palette='viridis') #Counplot of Academic Level
plt.title('Counplot of Academic Level') #Title of the plot
plt.xlabel('Academic Level') #level of x-axis
plt.ylabel('Count') #level of y-axis
plt.show()

In [ ]:
df.groupby('academic_level')['exam_score'].mean() # Calculates the average exam score for each academic level

In [ ]:
group = df.groupby('academic_level')['exam_score'].mean().reset_index() # Groups the data by academic level and computes the mean exam score for each level
plt.figure(figsize=(10,6)) #Size of the plot
sns.barplot(data=group, x='academic_level', y='exam_score',palette='plasma') #craete a barplot showing average exam scores across academic level
plt.title('Average Exam Score by Academic Level') #Title of the plot
plt.xlabel('Academic Level') # x-level of the plot
plt.ylabel('Average Exam Acore') #y-level of the plot
plt.show() #Display of the plot


In [ ]:
df.groupby(['gender','academic_level'])['exam_score'].mean() # Calculates the average exam score for each combination of gender and academic level

In [ ]:
df['internet_quality'].value_counts() # Count of internet quality

In [ ]:
plt.figure(figsize=(10,6)) # set the figure size
sns.countplot(data=df,x='internet_quality',palette='magma') #Counplot of internet_quality
plt.title('Counplot of internet quality') #Title of the plot
plt.xlabel('Internet_Quality') #level of x-axis
plt.ylabel('Count') #level of y-axis
plt.show()

In [ ]:
numerical_cols = ['exam_score', 'study_hours', 'sleep_hours', 'productivity_score'] # Selects key numerical columns for distribution analysis

for col in numerical_cols:
    plt.figure(figsize=(8,5))  # Creates a new figure for each numerical feature
    sns.histplot(df[col], bins=15, kde=True, color='lightgreen')     # Plots a histogram with 15 bins and a KDE curve to show the distribution
    plt.title(f'Distribution of {col}')   # Sets the title dynamically for each feature
    plt.xlabel(col)   # Labels the x-axis with the feature name
    plt.ylabel('Frequency')     # Labels the y-axis as frequency/
    plt.show() #Display the plot


In [ ]:
plt.figure(figsize=(10,6)) # Creates a figure with specified width and height
sns.histplot(data=df, x='exam_score', hue='gender', bins=15, kde=True, palette='Set2', multiple='stack') # Plots stacked histograms of exam score distribution for each gender with a KDE curve
plt.title('Exam Score Distribution by Gender') #Title of the plot
plt.xlabel('Exam Score') #x-axis of the plot
plt.ylabel('Count') #y-axis of the plot
plt.show() #Display of the plot

In [ ]:
le=LabelEncoder() #Intialize the LabelEncoder
df['academic_level']=le.fit_transform(df['academic_level']) # Encode the feature academic_level
df['internet_quality'] = le.fit_transform(df['internet_quality']) # encode the feature internet quality

In [ ]:
ohe = OneHotEncoder(sparse_output=False) # Initializes the OneHotEncoder and sets output to dense array instead of sparse matrix
encoded = ohe.fit_transform(df[['gender']]) # Fits the encoder on the gender column and transforms it into one-hot encoded numerical values

encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(['gender'])
) # Creates a DataFrame from the encoded array with proper column names

df = pd.concat([df.drop('gender', axis=1), encoded_df], axis=1) # Creates a DataFrame from the encoded array with proper column names

In [ ]:
x=df.drop(['exam_score'],axis=1) #independent variables
y=df['exam_score'] #target variable

In [ ]:
corr=x.corrwith(y) #calculation the correlation between target variable and independent variable
print(corr)

In [ ]:
corr_selected = corr[abs(corr) > 0.05] #Select only features with correlation magnitude > 0.05
corr_selected.sort_values(ascending=False) # display result

In [ ]:
import statsmodels.api as sm # Import the statsmodels library
x_const=sm.add_constant(x) # Add a constant (intercept) to the independent variables
model=sm.OLS(y,x_const) #Create an OLS regression model with dependent variable y and independent variables x_const
result=model.fit() ## Fit the model to the data
print(result.summary()) #Display the summary result

In [ ]:
ols_pval = result.pvalues.drop('const') # Get p-values of all coefficients from the OLS result and remove the intercept
ols_selected = ols_pval[ols_pval < 0.05].index.tolist() ## Select the variable names whose p-value is less than 0.05 (statistically significant)
ols_selected #display the selected feature by ols method

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
scaler=StandardScaler() #initialize the standardscaler
x_train=scaler.fit_transform(x_train) #fit and transform the training data
x_test=scaler.transform(x_test) #test data transform

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42) #Fit the Model
rf.fit(x, y)
importance = pd.Series(rf.feature_importances_, index=x.columns) #Get the importance
importance_sorted = importance.sort_values(ascending=False) #Sort feature importace in descending order
print(importance_sorted) #display the score

In [ ]:
importance_selected = importance[importance > 0.01] #Select features with importance greater than 0.01
print(importance_selected)  # Print the selected features and their importance

In [ ]:
final_features = list(set(ols_selected + corr_selected.index.tolist() + importance_selected.index.tolist())) #Combine all selected features from OLS p-values, correlation filtering, and feature importance
print("Selected ML features:", final_features) # Print the final list of features selected for the machine learning model

In [ ]:
x_new=df[final_features] #select only the importtant features

In [ ]:
lr=LinearRegression() # create a linear regression model
lr.fit(x_train,y_train) # fit the linear regression model in x_train and y_train
lr_pred=lr.predict(x_test) # make prediction of test data

In [ ]:
dt=DecisionTreeRegressor(random_state=42) # create a decision tree model
dt.fit(x_train,y_train) #fit the decision tree model
dt_predict=dt.predict(x_test) # make prediction of test data

In [ ]:
rf=RandomForestRegressor(n_estimators=100,random_state=42) # carete a randomforest model
rf.fit(x_train,y_train) # fit the randomforest tree model
rf_predict=rf.predict(x_test) # make prediction of test data

In [ ]:
svr=SVR(kernel='rbf') # create a support vector regression model
svr.fit(x_train,y_train) # fit support veator regression tree
svr_predict=svr.predict(x_test) # make prediction of test data

In [ ]:
xgb=XGBRegressor(n_estimators=100,random_state=42) # create xgbboost regreesion model
xgb.fit(x_train,y_train) #fit the xgb boost in
xgb_pred=xgb.predict(x_test) #make prediction of test data

In [ ]:
def evaluate(y_true,y_pred):
  return{
      "MAE":mean_absolute_error(y_true,y_pred),
      "RSME":np.sqrt(mean_squared_error(y_true,y_pred)),
      "R2":r2_score(y_true,y_pred)
  } # Define a function to evaluate model performance using common regression metrics

results = pd.DataFrame({
    "Linear Regression":evaluate(y_test,lr_pred),
    "Decision Tree": evaluate(y_test, dt_predict),
    "Random Forest": evaluate(y_test, rf_predict),
    "SVR": evaluate(y_test, svr_predict),
    "XGBoost": evaluate(y_test, xgb_pred)
}) ## Create a DataFrame to compare performance of multiple models

In [ ]:
results #Display the result

In [ ]:
plt.figure(figsize=(8,6)) # set the figure size

sns.barplot(
    x=results.columns,
    y=results.loc['RSME'],palette='viridis'
) # Plot a bar chart comparing models based on RMSE

plt.title("Model Comparison Based on RMSE") # set the title of the plot
plt.xlabel("Model") # set the label of x-axis
plt.ylabel("RMSE") #set the label of y-axis
plt.xticks(rotation=45) # Rotate x-axis labels for better readability
plt.grid(axis='y') # Add horizontal grid lines for easier comparison
plt.show() # display the plot

In [ ]:
input_data = (18, 'Other', 'High School', 2.21, 2.22, 2.1, 1.65, 2.55, 5.97, 6.05, 111, 339, 0, 0, 'Good', 3, 15.92, 37, 13.7, 1)

# Original column names from df after student_id was dropped
original_df_cols = ['age', 'gender', 'academic_level', 'study_hours', 'self_study_hours', 'online_classes_hours', 'social_media_hours', 'gaming_hours', 'sleep_hours', 'screen_time_hours', 'exercise_minutes', 'caffeine_intake_mg', 'part_time_job', 'upcoming_deadline', 'internet_quality', 'mental_health_score', 'focus_index', 'burnout_level', 'productivity_score', 'exam_score']

# Create a DataFrame for the single input sample
input_df_raw = pd.DataFrame([input_data], columns=original_df_cols)

# Separate features from target (exam_score is the last element in input_data)
input_features_to_process = input_df_raw.drop(columns=['exam_score'])

# Create a new LabelEncoder specifically for 'academic_level' and fit it with the known classes
# This bypasses the issue of the global 'le' being overwritten.
le_academic = LabelEncoder()
le_academic.fit(['High School', 'Postgraduate', 'Undergraduate']) # Fit with known academic levels
input_features_to_process['academic_level'] = le_academic.transform(input_features_to_process['academic_level'])

# Use the existing 'le' for 'internet_quality' as it was last fitted on it
input_features_to_process['internet_quality'] = le.transform(input_features_to_process['internet_quality'])

# Apply One-Hot Encoding for 'gender'
gender_encoded = ohe.transform(input_features_to_process[['gender']])
gender_encoded_df = pd.DataFrame(gender_encoded, columns=ohe.get_feature_names_out(['gender']), index=input_features_to_process.index)
input_features_processed = pd.concat([input_features_to_process.drop('gender', axis=1), gender_encoded_df], axis=1)

# Ensure the column order matches 'x' (the data used to train the scaler and models)
expected_feature_order = x.columns.tolist()
input_final_features = input_features_processed[expected_feature_order]

# Scale features using the fitted scaler
input_data_scaled = scaler.transform(input_final_features)

# Predict using the trained Linear Regression model
predicted_score = lr.predict(input_data_scaled)
print("Predicted Exam Score:", predicted_score[0])